In [7]:
import requests
import yfinance as yf
import json
import time
import os

API_KEY = "PFNCG4FDSWHLTRV5"
QUARTER_MONTH = {"Q1":"02", "Q2":"05", "Q3":"08", "Q4":"11"}
os.makedirs("data/transcripts", exist_ok=True)

tickers = [
    "AAPL", "MSFT", "GOOGL", "NVDA", "META",
    "JPM", "BAC", "GS", "V", "MA",
    "JNJ", "PFE", "UNH", "ABBV", "MRK",
    "AMZN", "TSLA", "MCD", "NKE", "SBUX",
    "XOM", "CVX", "HD", "HON", "UPS"
]

quarters = ["2022Q3", "2022Q4", "2023Q1", "2023Q2", "2023Q3", "2023Q4", "2024Q1", "2024Q2", "2024Q3"]


In [10]:
MAX_CALLS = 25
calls_made = 0

for ticker in tickers:
    if calls_made >= MAX_CALLS:
        print("Hit daily limit, run again tomorrow!")
        break
    for quarter in quarters:
        if calls_made >= MAX_CALLS:
            break
        
        filename = f"data/transcripts/{ticker}_{quarter}.json"
        if os.path.exists(filename):
            print(f"Already have {ticker} {quarter}, skipping")
            continue
        
        # fetch transcript
        params = {"function":"EARNINGS_CALL_TRANSCRIPT","symbol":ticker,"quarter":quarter,"apikey":API_KEY}
        transcript_data = requests.get("https://www.alphavantage.co/query", params=params).json()
        
        # CHECK BEFORE SAVING
        if "Information" in transcript_data or "Note" in transcript_data or "transcript" not in transcript_data:
            print(f"✗ {ticker} {quarter}: API limit hit, stopping for today")
            break
        
        # fetch return
        year, q = quarter[:4], quarter[4:]
        date = f"{year}-{QUARTER_MONTH[q]}-01"
        hist = yf.Ticker(ticker).history(start=date, period="10d")
        return_pct = (float(hist["Close"].iloc[3]) - float(hist["Close"].iloc[0])) / float(hist["Close"].iloc[0]) * 100
        
        # save
        transcript_data["return_pct"] = return_pct
        transcript_data["earnings_date"] = date
        transcript_data["symbol"] = ticker
        transcript_data["quarter"] = quarter
        with open(filename, "w") as f:
            json.dump(transcript_data, f)
        
        print(f"✓ {ticker} {quarter}: {return_pct:.2f}%")
        calls_made += 1
        time.sleep(13)

print(f"Done! Collected {calls_made} transcripts today.")

Already have AAPL 2022Q3, skipping
Already have AAPL 2022Q4, skipping
Already have AAPL 2023Q1, skipping
Already have AAPL 2023Q2, skipping
Already have AAPL 2023Q3, skipping
Already have AAPL 2023Q4, skipping
Already have AAPL 2024Q1, skipping
Already have AAPL 2024Q2, skipping
Already have AAPL 2024Q3, skipping
Already have MSFT 2022Q3, skipping
Already have MSFT 2022Q4, skipping
Already have MSFT 2023Q1, skipping
Already have MSFT 2023Q2, skipping
Already have MSFT 2023Q3, skipping
Already have MSFT 2023Q4, skipping
Already have MSFT 2024Q1, skipping
Already have MSFT 2024Q2, skipping
Already have MSFT 2024Q3, skipping
Already have GOOGL 2022Q3, skipping
Already have GOOGL 2022Q4, skipping
Already have GOOGL 2023Q1, skipping
Already have GOOGL 2023Q2, skipping
Already have GOOGL 2023Q3, skipping
Already have GOOGL 2023Q4, skipping
Already have GOOGL 2024Q1, skipping
Already have GOOGL 2024Q2, skipping
Already have GOOGL 2024Q3, skipping
Already have NVDA 2022Q3, skipping
Already hav

In [17]:
from pathlib import Path
import json

data_dir = Path("/gpfs/home/smaccall/earnings-call-prediction/notebooks/data/transcripts")

missing = []
for path in data_dir.glob("*.json"):
    with open(path) as f:
        d = json.load(f)
    if "return_pct" not in d:
        missing.append(path.name)
print(f"{len(missing)} files missing return_pct:")
print(missing)
missing = []
for path in data_dir.glob("*.json"):
    with open(path) as f:
        d = json.load(f)
    if "return_pct" not in d:
        missing.append(path.name)
print(f"{len(missing)} files missing return_pct:")
print(missing)

0 files missing return_pct:
[]
0 files missing return_pct:
[]


In [3]:
import json
from pathlib import Path

data_dir = Path("notebooks/data/transcripts")
deleted = 0
for f in data_dir.glob("*.json"):
    d = json.load(open(f))
    if "transcript" not in d:
        f.unlink()
        deleted += 1
        print(f"Deleted {f.name}")

print(f"\nDeleted {deleted} bad files")
print(f"Good files remaining: {len(list(data_dir.glob('*.json')))}")


Deleted 0 bad files
Good files remaining: 0
